### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [4]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



/tmp/ipykernel_9902/979371390.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/home/xievi/rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def process_all_pdf(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    #print(pdf_files)

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata 
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")
    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

#process all pdfs in the data directory
all_pdf_documents = process_all_pdf("../data")



Found 3 PDF files to process

Processing: Cal Poly Slo Decision Letter.pdf
 Loaded 1 pages

Processing: Alice B. Hansen Scholarship Form.pdf
 Loaded 1 pages

Processing: written assignment 3.pdf
 Loaded 5 pages

 Total documents loaded: 7


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'Technolutions', 'creator': 'Slate', 'creationdate': '2025-06-08T00:53:36-04:00', 'author': '', 'title': 'Cal Poly', 'moddate': '2025-06-08T00:53:36-04:00', 'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cal Poly Slo Decision Letter.pdf', 'file_type': 'pdf'}, page_content="1 GRAND AVENUE • SAN LUIS OBISPO • CALIFORNIA • 934 07                                       CALPOLY.EDU  \n \n \nApril 1, 2025 \n \nVictor Xie \n1516 Buena Vista Ave Apt A \nAlameda, CA 94501-1218\nCongratulations Victor,\nOur answer is, yes! You did it! You applied, we accepted and so it is my privilege to offer you\nconditional admission to the fall 2025 quarter at Cal Poly, where Learn by Doing has inspired and\nguided students toward lifelong success since 1901. (And where 95% of Mustangs are employed or in\ngraduate school within 9 months of graduation.)\nIn the Computer Science major, you will apply your 

In [7]:
###Text splitting (get into chunks)

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better Rag Performances"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators= ["\n\n", "\n", " ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [8]:
chunks = split_documents(all_pdf_documents)
chunks

Split 7 documents into 9 chunks

Example chunk:
Content: 1 GRAND AVENUE • SAN LUIS OBISPO • CALIFORNIA • 934 07                                       CALPOLY.EDU  
 
 
April 1, 2025 
 
Victor Xie 
1516 Buena Vista Ave Apt A 
Alameda, CA 94501-1218
Congratul...
Metadata: {'producer': 'Technolutions', 'creator': 'Slate', 'creationdate': '2025-06-08T00:53:36-04:00', 'author': '', 'title': 'Cal Poly', 'moddate': '2025-06-08T00:53:36-04:00', 'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cal Poly Slo Decision Letter.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Technolutions', 'creator': 'Slate', 'creationdate': '2025-06-08T00:53:36-04:00', 'author': '', 'title': 'Cal Poly', 'moddate': '2025-06-08T00:53:36-04:00', 'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cal Poly Slo Decision Letter.pdf', 'file_type': 'pdf'}, page_content="1 GRAND AVENUE • SAN LUIS OBISPO • CALIFORNIA • 934 07                                       CALPOLY.EDU  \n \n \nApril 1, 2025 \n \nVictor Xie \n1516 Buena Vista Ave Apt A \nAlameda, CA 94501-1218\nCongratulations Victor,\nOur answer is, yes! You did it! You applied, we accepted and so it is my privilege to offer you\nconditional admission to the fall 2025 quarter at Cal Poly, where Learn by Doing has inspired and\nguided students toward lifelong success since 1901. (And where 95% of Mustangs are employed or in\ngraduate school within 9 months of graduation.)\nIn the Computer Science major, you will apply your 

### embedding and vectorStoreDB

In [73]:
import numpy as np
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [74]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model successfully loaded. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not self.model:
            raise ValueError("Model Not Loaded")
        print(f"Generating  embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

##initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8617.69it/s]


Model successfully loaded. Embedding dimension: 384


### VectorStore

In [75]:
class VectorStore:
    """Mangaes document embeddings in a ChromaDB vector store """
    def __init__(self, collection_name = "pdf_documents", persist_directory = "../data/vector_store"):
        """Initialize the vector store"""

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok= True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for Rag"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vectore store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add doucuments and their embeddings to the vector store"""

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print (f"Adding {len(documents)} documents to vector store...")
    
        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            #Document content
            documents_text.append(doc.page_content)
            
            #Embedding
            embeddings_list.append(embedding.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,                
            )
            print(f"Sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vectore store: {e}")
            raise


vectorstore = VectorStore()
vectorstore
            

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 18


In [76]:
chunks

[Document(metadata={'producer': 'Adobe Acrobat DC Paper Capture Plug-in', 'creator': 'Adobe Scan for iOS 25.06.02', 'creationdate': '2025-06-08T04:55:48+00:00', 'moddate': '2025-06-08T04:55:48+00:00', 'source': '../data/pdf/Alice B. Hansen Scholarship Form.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Alice B. Hansen Scholarship Form.pdf', 'file_type': 'pdf'}, page_content='r \nAlice B. Hansen Scholarship Fund \nat the East Biry Community Foundation \nStudent Information Form \nEmail to scholarships @eastbaycf.org by June 30th: \n1) This Student Information Form with accurate information, completed and signed. \n-\nEAST~~ \nBAYil -\n2) A copy of the enrollment form or letter of acceptance from your college or university. \nI \nIV mm \nJ. I veri~ that I am pursuing an Asso ciate or Bachelor\'s Degree at a college or \nersitv in California. in accordance with the terms of this sch olarshio. \nStudent Name: \nPermanent Mailing Address: I S" / \\, \nHome Phone # (if

In [77]:
### convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate the embeddings 
embeddings = embedding_manager.generate_embeddings(texts)
embeddings

#store in the vector datavase
vectorstore.add_documents(chunks,embeddings)

Generating  embeddings for 9 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.40it/s]

Generated embeddings with shape: (9, 384)
Adding 9 documents to vector store...
Sucessfully added 9 documents to vector store
Total documents in collection: 27


### Retriever PipeLine From VectorStore

In [78]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever 
        
        Args: 
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings 
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    
    def retrieve(self, query:str, top_k:int = 5, score_threshold: float = 0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

            for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                #convert distance to similarity score (ChromaDB uses cosine distance)
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id':doc_id,
                        'content': document,
                        'metadata': metadata,
                        'similarity_score':similarity_score,
                        'distance':distance,
                        'rank':i + 1
                    })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        

rag_retriever = RAGRetriever(vectorstore, embedding_manager)


In [79]:
rag_retriever.retrieve("was I accepted into a cal  ")

Retrieving documents for query: 'was I accepted into a cal  '
Top K: 5, Score threshold: 0.0
Generating  embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 116.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)
No documents found


[{'id': 'doc_64fe9294_1',
  'content': "problem-solvers and a community of scholars dedicated to helping you reach your fullest potential,\nwhile pursuing solutions to humanity's pressing challenges.\nExciting! Join our Mustang family. Anxious? Reach out. We want to make the transition to university\nlife a smooth one.\nAgain, congratulations. Welcome to Cal Poly, where you have every opportunity to Learn by Doing,\nand to make a successful life for yourself and a brighter future for all.\nMelissa L. Furlong \nExecutive Director \nAdmissions and Enrollment Development \ncalpoly.edu/admissions\n \nPlease note your current tuition residency status is Resident of California.",
  'metadata': {'page': 0,
   'page_label': '1',
   'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf',
   'total_pages': 1,
   'file_type': 'pdf',
   'doc_index': 1,
   'content_length': 629,
   'source_file': 'Cal Poly Slo Decision Letter.pdf',
   'moddate': '2025-06-08T00:53:36-04:00',
   'creationdate': '20

### Integration Vectordb Context pipeline With LLM output

In [80]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [81]:
answer = rag_simple("who got into cal poly slo and who is victor xie",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'who got into cal poly slo and who is victor xie'
Top K: 3, Score threshold: 0.0
Generating  embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
No documents found
No relevant context found to answer the question.


### Enhanced RAG pipeline features


In [82]:
# --- Enhanced RAG pipeline features ---

def rag_advanced(query,retriever,llm,top_k=5,min_score=0.2,return_context=False):
    """
    RAG pipeline with extra features:
    Returns answers, sources, confidence score, and optionally full context."""

    results = retriever.retrieve(query,top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources':[],'confidence':0.0,'context':''}
    
    #Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:300]+'...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # prepare context and sources
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output


result = rag_advanced("Who was accepted into Cal Poly SLO", rag_retriever, llm, top_k=3, min_score=0.1,return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'Who was accepted into Cal Poly SLO'
Top K: 3, Score threshold: 0.1
Generating  embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.25it/s]

Generated embeddings with shape: (1, 384)


Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
No documents found
Answer: The student who received the message (i.e., “you”) was accepted into Cal Poly SLO.
Sources: [{'source': 'Cal Poly Slo Decision Letter.pdf', 'page': 0, 'score': 0.1634448766708374, 'preview': "problem-solvers and a community of scholars dedicated to helping you reach your fullest potential,\nwhile pursuing solutions to humanity's pressing challenges.\nExciting! Join our Mustang family. Anxious? Reach out. We want to make the transition to university\nlife a smooth one.\nAgain, congratulations..."}, {'source': 'Cal Poly Slo Decision Letter.pdf', 'page': 0, 'score': 0.1634448766708374, 'preview': "problem-solvers and a community of scholars dedicated to helping you reach your fullest potential,\nwhile pursuing solutions to humanity's pressing challenges.\nExciting! Join our Mustang family. Anxious? Reach out. We want to make the transition to u